In [ ]:
import torch
from torch import nn
from d2l import torch as d2l
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.optim as optim

In [ ]:
## THIS CODE IS TO CREATE THE TENSOR FOR THE TRAINING DATA ##

je_df = pd.read_csv('../../Data/Final/TT Split/je_train_long.csv')

# Pivot to create a time series for each (x, y)
je_pivot = je_df.pivot(index=['x', 'y'], columns='Date', values='Value')
#print (je_pivot)

# Reset index to keep (x, y) as columns
je_pivot = je_pivot.reset_index()

# Convert time columns back into a NumPy array
ts_je = je_pivot.iloc[:, 2:].values  # Ignore first two columns (x, y)
loc_je = je_pivot.iloc[:, :2].values  # Store coordinates

# Standardization with Z-score normalisation
je_mean_LST = ts_je.mean()
je_std_LST = ts_je.std()
ts_je = (ts_je - je_mean_LST) / je_std_LST 

# Convert to PyTorch tensor wiith extra dimension for LST
X_train_tensor_je = torch.tensor(ts_je, dtype=torch.float32).unsqueeze(-1)
#print(X_train_tensor.shape)

X_train_je = X_train_tensor_je
y_train_je = X_train_tensor_je[:, 1:, :]

In [ ]:
## THIS CODE IS TO DEFINE THE LSTM MODEL ##

class LSTMPredictor(nn.Module):
    def __init__(self, input_size=1, hidden_size=64, num_layers=3, output_size=1):
        super(LSTMPredictor, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)  # Fully connected layer

    def forward(self, x):
        lstm_out, _ = self.lstm(x)  # LSTM output
        out = self.fc(lstm_out)  # Fully connected layer for final prediction
        return out

In [ ]:
## THIS CODE IS TO TRAIN THE MODEL ##
torch.manual_seed(5188)

model = LSTMPredictor() # Model
criterion = nn.MSELoss() # Loss function to track performance over epochs
optimizer = optim.Adam(model.parameters(), lr=0.001) # Adam optimizer (can be switched)

# Training loop
num_epochs = 100
for epoch in range(num_epochs):
    model.train()
    optimizer.zero_grad()

    predictions = model(X_train_je)

    loss = criterion(predictions[:, :-1, :], y_train_je)
    loss.backward()
    optimizer.step()

    if epoch % 10 == 0: # Print loss every 10 epochs
        print(f"Epoch {epoch}/{num_epochs}, Loss: {loss.item():.4f}")

Epoch 0/100, Loss: 1.0083
Epoch 10/100, Loss: 0.9913
Epoch 20/100, Loss: 0.9562
Epoch 30/100, Loss: 0.9323
Epoch 40/100, Loss: 0.9088
Epoch 50/100, Loss: 0.9028
Epoch 60/100, Loss: 0.8982
Epoch 70/100, Loss: 0.8929
Epoch 80/100, Loss: 0.8848
Epoch 90/100, Loss: 0.8729


In [ ]:
## THIS CODE IS FOR FORECASTING##

model.eval()
torch.manual_seed(5188)

with torch.no_grad():
    X_pred_je = model(X_train_je)  # Forecast next steps
    X_pred_je= X_pred_je[:, -12:, :]  # Extract last 12 bimonthly periods (2023-2024)

# Convert predictions to a NumPy array
X_pred_je_np = X_pred_je.squeeze().cpu().numpy()

# **Denormalize predictions**
X_pred_je_np = (X_pred_je_np * je_std_LST) + je_mean_LST  # Convert back to original from scaled values

# Define bimonthly periods
bimonthly_periods = [
    "Jan-Feb 2023", "Mar-Apr 2023", "May-Jun 2023", "Jul-Aug 2023", "Sep-Oct 2023", "Nov-Dec 2023",
    "Jan-Feb 2024", "Mar-Apr 2024", "May-Jun 2024", "Jul-Aug 2024", "Sep-Oct 2024", "Nov-Dec 2024"
]

# Create a long-format DataFrame
je_pred_df = pd.DataFrame({
    "x": np.repeat(loc_je[:, 0], len(bimonthly_periods)),  # Use stored locations
    "y": np.repeat(loc_je[:, 1], len(bimonthly_periods)),  
    "Date": bimonthly_periods * len(loc_je),
    "Predicted_LST": X_pred_je_np.flatten()  # Store denormalized values
})

je_pred_df.to_csv('je_pred_long.csv', index=False) # Save to CSV for RMSE

In [ ]:
## THIS CODE IS TO FIND THE RMSE BETWEEN TRUE AND PREDICTED LST ##

true_df = pd.read_csv("je_pred_long.csv")
pred_df = pd.read_csv("../../Data/Final/TT Split/je_test_long.csv")

merged_df = true_df.merge(pred_df, on=["x", "y", "Date"], suffixes=("_true", "_pred"))
print(merged_df.head())

# Compute RMSE
rmse = np.sqrt(np.mean((merged_df["Predicted_LST"] - merged_df["Value"]) ** 2))
print(f"RMSE between true and predicted LST for Jurong East is: {rmse:.4f}")

            x         y          Date  Predicted_LST      Value
0  103.964566  1.350459  Jan-Feb 2023      25.844383  19.300148
1  103.964566  1.350459  Mar-Apr 2023      26.107073  21.802744
2  103.964566  1.350459  May-Jun 2023      26.135025  23.389268
3  103.964566  1.350459  Jul-Aug 2023      26.061527  22.872917
4  103.964566  1.350459  Sep-Oct 2023      25.944980  27.238957
RMSE between true and predicted LST for Changi is: 3.4058
